Load clean ERCOT data

In [159]:
import pandas as pd
import duckdb

conn = duckdb.connect("/Users/mikemueller/GitHub/ercot_load_forecasting/data/database/ercot.duckdb")

conn.sql("SET TimeZone = 'UTC'")

df = conn.sql('''SELECT * FROM ercot_load_clean''').df()

df.columns


Index(['local_timestamp', 'utc_timestamp', 'day_of_year', 'day_of_week',
       'is_weekend', 'is_holiday', 'fallback_flag', 'dst_flag', 'COAST',
       'EAST', 'FAR_WEST', 'NORTH', 'NORTH_C', 'SOUTHERN', 'SOUTH_C', 'WEST',
       'ERCOT'],
      dtype='str')

Missing values

00:00 local time is missing because the source data is missing. This is the only missing data in the dataset out of 22 years, so I am not going to impute any values. This may cost us some additional rows due to lag features (e.g. 00 LST Nov 8 24-hr lag; 00 LST Nov 14 168-hr lag). But I think it's best to leave it missing for now.

In [155]:
df[df['ERCOT'].isna()]
df.columns
df['local_timestamp']=pd.to_datetime(df['local_timestamp'])
df['year']=df['local_timestamp'].dt.year
df['month']=df['local_timestamp'].dt.month
missing_rows = df.isna().any(axis=1)

i = df.index[missing_rows][0]

df.loc[i-2:i+2]

,local_timestamp,COAST,EAST,FAR_WEST,NORTH,NORTH_C,SOUTHERN,SOUTH_C,WEST,ERCOT,dst_flag,day_of_week,is_weekend,is_holiday,fallback_flag,utc_timestamp,day_of_year,year,month
112653,2016-11-06 22:00:00,10847.035095,1189.700214,2038.580496,670.050687,10339.590836,3299.416020,5519.916371,963.305845,34867.595565,False,0,True,False,False,2016-11-07 05:00:00+00:00,311,2016,11
112654,2016-11-06 23:00:00,10138.494300,1075.237940,1975.049230,630.403100,9586.779800,3064.395300,5058.636620,906.891170,32435.887460,False,0,True,False,False,2016-11-07 06:00:00+00:00,311,2016,11
112655,2016-11-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1,False,False,False,2016-11-07 07:00:00+00:00,312,2016,11
112656,2016-11-07 01:00:00,8961.697811,979.187263,1884.424615,577.562032,8236.213944,2629.115180,4367.111544,832.889076,28468.201466,False,1,False,False,False,2016-11-07 08:00:00+00:00,312,2016,11
112657,2016-11-07 02:00:00,8719.073733,947.195502,1863.183132,566.703350,7924.049415,2512.061383,4176.500639,808.787183,27517.554336,False,1,False,False,False,2016-11-07 09:00:00+00:00,312,2016,11


Found May 2026 and June 1 2026 00 Local time had duplicate rows. Deduplicated those in ETL.

In [156]:
df.columns
df[df['dst_flag']==True]
df['local_timestamp'].isna().sum()
(df['local_timestamp'].count())/24
df['local_timestamp'].is_monotonic_increasing #False
df['local_timestamp'].duplicated().sum() #766
df[df['local_timestamp'].duplicated(keep=False)]
dup_mask = df['local_timestamp'].duplicated(keep=False)

df.loc[dup_mask, ['local_timestamp', 'dst_flag']]
df['local_timestamp'].value_counts().head(30)

df.duplicated().sum()
df[df.duplicated(keep=False)].sort_values('local_timestamp')


df = df.drop_duplicates()
print(df.duplicated().sum())
print(df['local_timestamp'].duplicated().sum())

df[df['local_timestamp'].duplicated(keep=False)].groupby(
    df['local_timestamp'].dt.year
).size()

df.loc[
    df['local_timestamp'].duplicated(keep=False),
    'local_timestamp'
].sort_values()

df[df['year'] != 2026]

#print(f"Exact duplicates remaining: {df.duplicated().sum()}")
#print(
#    f"Duplicate local timestamps remaining: "
#    f"{df['local_timestamp'].duplicated().sum()}"
#)

df.groupby('year')['ERCOT'].size()
df.columns

df_etl.columns
df[['local_timestamp']].head()
#conn.close()




0
22


,local_timestamp
0,2004-01-01 01:00:00
1,2004-01-01 02:00:00
2,2004-01-01 03:00:00
3,2004-01-01 04:00:00
4,2004-01-01 05:00:00


In [157]:
df_etl.columns
fallback_positions = df.index[df['fallback_flag']].tolist()
#i = fallback_positions[0]
diff = df['local_timestamp'].diff()

df.loc[
    diff.notna() & (diff != pd.Timedelta(hours=1)),
    ['local_timestamp', 'utc_timestamp']
]

df['utc_timestamp'].diff().value_counts()

df['local_timestamp'].is_monotonic_increasing
df['utc_timestamp'].is_monotonic_increasing

True

In [158]:
df.columns
conn.close()